<a href="https://colab.research.google.com/github/rodrigoiyg27/TopicosEspeciales2026/blob/main/Clase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
url= ""
df = pd.read_csv(url)
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Exploración inicial

In [ ]:
print("Filas y columnas", df.shape)
print("Nombres de columnas", df.columns)
print("Tipos de datos", df.dtypes)
print("Valores nulos", df.isnull().sum())

Filas y columnas (10000, 8)
Nombres de columnas Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')
Tipos de datos Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object
Valores nulos Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


Detección de valores nulos

In [ ]:
# .isna() devuelve una matriz booleana; .sum() cuenta los True por columna
nulos_por_columna = df.isna().sum()
porcentaje_nulos = (df.isna().mean() * 100).round(2)

resumen_nulos = pd.DataFrame({
    "nulos": nulos_por_columna,
    "porcentaje_%": porcentaje_nulos
}).sort_values("porcentaje_%", ascending=False)

print(resumen_nulos)

                  nulos  porcentaje_%
Location           3265         32.65
Payment Method     2579         25.79
Item                333          3.33
Price Per Unit      179          1.79
Total Spent         173          1.73
Transaction Date    159          1.59
Quantity            138          1.38
Transaction ID        0          0.00


Tratamiento de valores nulos

Imputar variables numéricas: decidir media vs. mediana con sesgo

In [ ]:
columnas_numericas = ["Quantity", "Price Per Unit", "Total Spent"]

for col in columnas_numericas:
   # pd.to_numeric fuerza la columna a tipo numérico.
    # errors="coerce" convierte cualquier texto no numérico (p. ej. "ERROR",
    # "UNKNOWN") en NaN, en vez de detener la ejecución con un error.
    df[col] = pd.to_numeric(df[col], errors="coerce")
    media_col = df[col].mean()
    mediana_col = df[col].median()
    sesgo = df[col].skew()  # mide la asimetría de la distribución

    # Regla: |skew| < 0.5 -> distribución simétrica -> media
    #        |skew| >= 0.5 -> distribución sesgada / outliers -> mediana
    if abs(sesgo) < 0.5:
        valor_imputacion = media_col
        criterio = "media"
    else:
        valor_imputacion = mediana_col
        criterio = "mediana"

    df[col] = df[col].fillna(valor_imputacion)
    print(f"{col}: skew={sesgo:.2f} -> se imputó con {criterio} = {valor_imputacion:.2f}")

Quantity: skew=-0.01 -> se imputó con media = 3.03
Price Per Unit: skew=0.00 -> se imputó con media = 2.95
Total Spent: skew=0.82 -> se imputó con mediana = 8.00


Imputar categóricas con bajo % de nulos usando la moda

In [ ]:
moda_item = df["Item"].mode(dropna=True)[0]
df["Item"] = df["Item"].fillna(moda_item)
print(f"Item: imputado con moda = {moda_item}")

Item: imputado con moda = Juice


 Nulos altos en categóricas: NO se imputa con moda

In [ ]:
# Payment Method (25.79%) y Location (32.65%) tienen demasiados nulos.
# Imputar con la moda distorsionaría fuertemente esas columnas (le estaríamos
# "inventando" el método de pago o el lugar a 1 de cada 3-4 transacciones).
# Preferible documentar la ausencia como una categoría explícita.
df["Payment Method"] = df["Payment Method"].fillna("No especificado")
df["Location"] = df["Location"].fillna("No especificado")

Eliminar filas solo si el campo no es crítico

> Agregar bloque entrecomillado



In [ ]:
# Transaction Date es necesaria para cualquier análisis de tendencias en el
# tiempo. Con solo 1.59% de nulos, es preferible descartar esas
# filas en vez de inventar una fecha.
df = df.dropna(subset=["Transaction Date"])

Verificación final de nulos

In [ ]:
print("\nNulos restantes por columna:")
print(df.isna().sum())


Nulos restantes por columna:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


Duplicados

In [ ]:
# Duplicados exactos: todas las columnas idénticas
duplicados_exactos = df.duplicated().sum()
print(f"Filas totalmente duplicadas: {duplicados_exactos}")

# Duplicados por columna clave: Transaction ID repetido
duplicados_id = df.duplicated(subset=["Transaction ID"]).sum()
print(f"Transaction ID repetidos: {duplicados_id}")

# Si hay duplicados por ID, es útil verlos para decidir qué hacer
if duplicados_id > 0:
    ids_repetidos = df[df.duplicated(subset=["Transaction ID"], keep=False)]
    print(ids_repetidos.sort_values("Transaction ID"))

Filas totalmente duplicadas: 0
Transaction ID repetidos: 0


Agregamos un duplicado para simular situación

In [ ]:
fila_duplicada = df.iloc[[0]]
df_demo = pd.concat([df, fila_duplicada], ignore_index=True)

print("\n[Demo] Duplicados antes de limpiar:", df_demo.duplicated().sum())
df_demo = df_demo.drop_duplicates(keep="first")
print("[Demo] Duplicados después de limpiar:", df_demo.duplicated().sum())


[Demo] Duplicados antes de limpiar: 1
[Demo] Duplicados después de limpiar: 0


Formatos inconsistentes

In [ ]:
# Inspeccionar valores únicos de cada columna de texto antes de normalizar
for col in ["Item", "Payment Method", "Location"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

--- Payment Method ---
Payment Method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

--- Location ---
Location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64


Ejemplo simulado

In [ ]:
# --- 1) Verificación de trim y normalización
for col in ["Item", "Payment Method", "Location"]:
    valores_unicos = df[col].dropna().unique()
    print(f"{col}: {len(valores_unicos)} valores únicos -> {sorted(valores_unicos)}")

#Ejercicio didáctico:
# Se usa solo con fines de aprendizaje; el dataset real no lo necesita.
ejemplo_sucio = pd.Series(["Coffee", "coffee", " Coffee ", "COFFEE"])
print("\n[Demo] Antes de normalizar:", ejemplo_sucio.nunique(), "valores únicos")

ejemplo_limpio = ejemplo_sucio.str.strip().str.title()
print("[Demo] Después de normalizar:", ejemplo_limpio.nunique(), "valores únicos")

Item: 10 valores únicos -> ['Cake', 'Coffee', 'Cookie', 'ERROR', 'Juice', 'Salad', 'Sandwich', 'Smoothie', 'Tea', 'UNKNOWN']
Payment Method: 5 valores únicos -> ['Cash', 'Credit Card', 'Digital Wallet', 'ERROR', 'UNKNOWN']
Location: 4 valores únicos -> ['ERROR', 'In-store', 'Takeaway', 'UNKNOWN']

[Demo] Antes de normalizar: 4 valores únicos
[Demo] Después de normalizar: 1 valores únicos


Formatos inconsistentes

In [ ]:
# Inspeccionar cómo llegan las fechas antes de convertirlas
print(df["Transaction Date"].dtype)          # revisa si Pandas la reconoce como fecha o como texto
print(df["Transaction Date"].head(10))        # muestra formato(s) presentes
print(df["Transaction Date"].sample(10))      # muestra una muestra aleatoria, por si el head() no es representativo

object
0    2023-09-08
1    2023-05-16
2    2023-07-19
3    2023-04-27
4    2023-06-11
5    2023-03-31
6    2023-10-06
7    2023-10-28
8    2023-07-28
9    2023-12-31
Name: Transaction Date, dtype: object
8702    2023-07-03
2885    2023-05-15
8939    2023-03-22
797     2023-12-18
8935    2023-07-03
4523    2023-08-11
4246    2023-01-13
1882    2023-07-05
3633    2023-03-17
4657    2023-01-31
Name: Transaction Date, dtype: object


Convertir de objetc a date

In [ ]:
# --- Conversión de Transaction Date a datetime
fechas_antes = df["Transaction Date"].isna().sum()

# errors="coerce" convierte cualquier valor no interpretable como fecha
# (incluyendo "ERROR", "UNKNOWN" o formatos corruptos) en NaT (Not a Time),

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

fechas_despues = df["Transaction Date"].isna().sum()

print(f"Nulos antes de convertir: {fechas_antes}")
print(f"Nulos después de convertir: {fechas_despues}")
print(f"Fechas inválidas detectadas: {fechas_despues - fechas_antes}")

Nulos antes de convertir: 460
Nulos después de convertir: 460
Fechas inválidas detectadas: 0
